---

# **[Open-Source Project Feature Selection 1 Process]**

- 분석 단위: 오픈소스 repository
- **1차 selection** = repo JSON schema 기반 endpoint 후보 도출
- **2차 selection** = health indicator 논리 기반 endpoint pruning
- **3차 selection** = endpoint 내부 key selection (raw feature mining)
- **4차** = derived feature engineering

> **한 repository를 평가하기 위해 사용할 수 있는 repository-scoped endpoint 집합을 도출하는 단계**


---

## **[1st Selection]**

In [4]:
import os
import time
import requests
import pandas as pd
from datetime import datetime, timezone
from dotenv import load_dotenv

load_dotenv()
token = os.getenv("GITHUB_TOKEN")
HEADERS = {
    "Accept": "application/vnd.github+json",
}
if token:
    HEADERS["Authorization"] = f"Bearer {token}"

BASE_URL = "https://api.github.com"
FULL_NAME = "pandas-dev/pandas"

[Section 1] 분석 대상 repository 설정

In [9]:
OWNER = "pandas-dev"
REPO = "pandas"
FULL_NAME = f"{OWNER}/{REPO}"

repo_url = f"{BASE_URL}/repos/{FULL_NAME}"
print(repo_url)

https://api.github.com/repos/pandas-dev/pandas


[Section 2] 공통 함수: API 호출

In [10]:
def fetch_json(url, params=None, headers=HEADERS, sleep_sec=0.2):
    res = requests.get(url, headers=headers, params=params)
    res.raise_for_status()
    time.sleep(sleep_sec)
    return res.json()

[Section 3] repository 기본 응답 확인

In [11]:
repo_data = fetch_json(repo_url)

print(type(repo_data))
print(len(repo_data))

<class 'dict'>
86


[Section 4] JSON 값 타입 판별 함수

In [15]:
def get_value_type(value):
    if value is None:
        return "NoneType"
    elif isinstance(value, bool):
        return "bool"
    elif isinstance(value, int):
        return "int"
    elif isinstance(value, float):
        return "float"
    elif isinstance(value, str):
        return "str"
    elif isinstance(value, dict):
        return "dict"
    elif isinstance(value, list):
        return "list"
    else:
        return type(value).__name__

[Section 5] GitHub URL / 링크 / 템플릿 여부 판별 함수

In [16]:
import re

def is_github_api_url(value):
    return isinstance(value, str) and value.startswith("https://api.github.com/")

def is_web_url(value):
    return isinstance(value, str) and value.startswith("http")

def has_url_template(value):
    return isinstance(value, str) and ("{" in value and "}" in value)

def clean_github_url_template(url):
    if not isinstance(url, str):
        return url
    return re.sub(r"\{.*?\}", "", url)

[Section 6] repo JSON 전체를 재귀적으로 순회하며 
- path / dtype / example / url 여부를 기록

In [17]:
def walk_json(obj, parent_path=""):
    rows = []

    if isinstance(obj, dict):
        for key, value in obj.items():
            current_path = f"{parent_path}.{key}" if parent_path else key
            value_type = get_value_type(value)

            row = {
                "path": current_path,
                "key": key,
                "dtype": value_type,
                "example_value": str(value)[:120],
                "is_github_api_url": is_github_api_url(value),
                "is_web_url": is_web_url(value),
                "has_url_template": has_url_template(value),
            }
            rows.append(row)

            if isinstance(value, (dict, list)):
                rows.extend(walk_json(value, current_path))

    elif isinstance(obj, list):
        # 리스트 자체도 하나의 구조 단위로 기록하고
        # 내부 원소 구조를 보기 위해 일부 샘플만 재귀 순회
        for idx, item in enumerate(obj[:10]):
            current_path = f"{parent_path}[{idx}]"
            value_type = get_value_type(item)

            row = {
                "path": current_path,
                "key": f"[{idx}]",
                "dtype": value_type,
                "example_value": str(item)[:120],
                "is_github_api_url": is_github_api_url(item),
                "is_web_url": is_web_url(item),
                "has_url_template": has_url_template(item),
            }
            rows.append(row)

            if isinstance(item, (dict, list)):
                rows.extend(walk_json(item, current_path))

    return rows

[Section 7] repo response 전체 schema inventory 생성

In [18]:
repo_schema_rows = walk_json(repo_data)
repo_schema_df = pd.DataFrame(repo_schema_rows)

repo_schema_df = repo_schema_df.drop_duplicates(
    subset=["path", "dtype", "example_value"]
).reset_index(drop=True)

print(repo_schema_df.shape)
display(repo_schema_df.head(100))

(135, 7)


,path,key,dtype,example_value,is_github_api_url,is_web_url,has_url_template
0,id,id,int,858127,False,False,False
1,node_id,node_id,str,MDEwOlJlcG9zaXRvcnk4NTgxMjc=,False,False,False
2,name,name,str,pandas,False,False,False
3,full_name,full_name,str,pandas-dev/pandas,False,False,False
4,private,private,bool,False,False,False,False
...,...,...,...,...,...,...,...
95,is_template,is_template,bool,False,False,False,False
96,web_commit_signoff_required,web_commit_signoff_required,bool,False,False,False,False
97,has_pull_requests,has_pull_requests,bool,True,False,False,False
98,pull_request_creation_policy,pull_request_creation_policy,str,all,False,False,False


[Section 8] dtype 기준으로 보기
- dict/list 구조가 어디 있는지 먼저 확인

In [19]:
repo_schema_df["dtype"].value_counts()

dtype
str         99
bool        16
int         13
dict         4
NoneType     2
list         1
Name: count, dtype: int64

[Section 9] dict / list path만 따로 확인

In [21]:
nested_structure_df = repo_schema_df[
    repo_schema_df["dtype"].isin(["dict", "list"])
].sort_values("path").reset_index(drop=True)

display(nested_structure_df)

,path,key,dtype,example_value,is_github_api_url,is_web_url,has_url_template
0,custom_properties,custom_properties,dict,{},False,False,False
1,license,license,dict,"{'key': 'bsd-3-clause', 'name': 'BSD 3-Clause ...",False,False,False
2,organization,organization,dict,"{'login': 'pandas-dev', 'id': 21206976, 'node_...",False,False,False
3,owner,owner,dict,"{'login': 'pandas-dev', 'id': 21206976, 'node_...",False,False,False
4,topics,topics,list,"['alignment', 'data-analysis', 'data-science',...",False,False,False


[Section 10] URL / 링크 / 템플릿 후보만 따로 확인
- endpoint 후보 도출의 출발점

In [22]:
url_candidate_df = repo_schema_df[
    (repo_schema_df["is_github_api_url"] == True) |
    (repo_schema_df["is_web_url"] == True) |
    (repo_schema_df["has_url_template"] == True)
].copy()

url_candidate_df["clean_url"] = url_candidate_df["example_value"].apply(clean_github_url_template)

url_candidate_df = url_candidate_df.sort_values("path").reset_index(drop=True)
display(url_candidate_df)

,path,key,dtype,example_value,is_github_api_url,is_web_url,has_url_template,clean_url
0,archive_url,archive_url,str,https://api.github.com/repos/pandas-dev/pandas...,True,True,True,https://api.github.com/repos/pandas-dev/pandas/
1,assignees_url,assignees_url,str,https://api.github.com/repos/pandas-dev/pandas...,True,True,True,https://api.github.com/repos/pandas-dev/pandas...
2,blobs_url,blobs_url,str,https://api.github.com/repos/pandas-dev/pandas...,True,True,True,https://api.github.com/repos/pandas-dev/pandas...
3,branches_url,branches_url,str,https://api.github.com/repos/pandas-dev/pandas...,True,True,True,https://api.github.com/repos/pandas-dev/pandas...
4,clone_url,clone_url,str,https://github.com/pandas-dev/pandas.git,False,True,False,https://github.com/pandas-dev/pandas.git
...,...,...,...,...,...,...,...,...
61,svn_url,svn_url,str,https://github.com/pandas-dev/pandas,False,True,False,https://github.com/pandas-dev/pandas
62,tags_url,tags_url,str,https://api.github.com/repos/pandas-dev/pandas...,True,True,False,https://api.github.com/repos/pandas-dev/pandas...
63,teams_url,teams_url,str,https://api.github.com/repos/pandas-dev/pandas...,True,True,False,https://api.github.com/repos/pandas-dev/pandas...
64,trees_url,trees_url,str,https://api.github.com/repos/pandas-dev/pandas...,True,True,True,https://api.github.com/repos/pandas-dev/pandas...


[Section 11] GitHub API endpoint 후보만 따로 정리
- repo response 내부에서 발견된 API endpoint inventory

In [23]:
endpoint_candidate_df = repo_schema_df[
    repo_schema_df["is_github_api_url"] == True
].copy()

endpoint_candidate_df["clean_url"] = endpoint_candidate_df["example_value"].apply(clean_github_url_template)

endpoint_candidate_df = endpoint_candidate_df[[
    "path", "key", "dtype", "example_value", "clean_url", "has_url_template"
]].drop_duplicates().sort_values("path").reset_index(drop=True)

display(endpoint_candidate_df)

,path,key,dtype,example_value,clean_url,has_url_template
0,archive_url,archive_url,str,https://api.github.com/repos/pandas-dev/pandas...,https://api.github.com/repos/pandas-dev/pandas/,True
1,assignees_url,assignees_url,str,https://api.github.com/repos/pandas-dev/pandas...,https://api.github.com/repos/pandas-dev/pandas...,True
2,blobs_url,blobs_url,str,https://api.github.com/repos/pandas-dev/pandas...,https://api.github.com/repos/pandas-dev/pandas...,True
3,branches_url,branches_url,str,https://api.github.com/repos/pandas-dev/pandas...,https://api.github.com/repos/pandas-dev/pandas...,True
4,collaborators_url,collaborators_url,str,https://api.github.com/repos/pandas-dev/pandas...,https://api.github.com/repos/pandas-dev/pandas...,True
5,comments_url,comments_url,str,https://api.github.com/repos/pandas-dev/pandas...,https://api.github.com/repos/pandas-dev/pandas...,True
6,commits_url,commits_url,str,https://api.github.com/repos/pandas-dev/pandas...,https://api.github.com/repos/pandas-dev/pandas...,True
7,compare_url,compare_url,str,https://api.github.com/repos/pandas-dev/pandas...,https://api.github.com/repos/pandas-dev/pandas...,True
8,contents_url,contents_url,str,https://api.github.com/repos/pandas-dev/pandas...,https://api.github.com/repos/pandas-dev/pandas...,True
9,contributors_url,contributors_url,str,https://api.github.com/repos/pandas-dev/pandas...,https://api.github.com/repos/pandas-dev/pandas...,False


In [25]:
repo_schema_df.to_csv("output/repo_response_schema_inventory.csv", index=False)
endpoint_candidate_df.to_csv("output/repo_response_endpoint_candidates.csv", index=False)

print("Saved:")
print("- repo_response_schema_inventory.csv")
print("- repo_response_endpoint_candidates.csv")

Saved:
- repo_response_schema_inventory.csv
- repo_response_endpoint_candidates.csv


### **[1st Selection — Repository Endpoint Selection Table]**

| endpoint_name     | selected | description                                                                |
| ----------------- | -------- | -------------------------------------------------------------------------- |
| url               | Yes      | Core repository metadata (license, visibility, size, default branch, etc.) |
| contributors_url  | Yes      | Contributor list and contribution distribution                             |
| commits_url       | Yes      | Commit history and activity timeline                                       |
| issues_url        | Yes      | Issue lifecycle, defect signals, responsiveness                            |
| pulls_url         | Yes      | Pull request collaboration and review process                              |
| comments_url      | Yes      | General discussion interactions                                            |
| issue_comment_url | Yes      | Issue-level interaction intensity                                          |
| issue_events_url  | Yes      | Issue state transitions and workflow signals                               |
| releases_url      | Yes      | Release management and versioning maturity                                 |
| tags_url          | Yes      | Version tag history                                                        |
| branches_url      | Yes      | Branch structure and operational workflow                                  |
| languages_url     | Yes      | Programming language composition                                           |
| labels_url        | Yes      | Issue categorization and governance signals                                |
| milestones_url    | Yes      | Planning and structured release cycles                                     |
| events_url        | Yes      | Repository-level activity events                                           |
| forks_url         | Yes      | Ecosystem spread and derivative development                                |
| stargazers_url    | Yes      | Popularity and adoption indicator                                          |
| subscribers_url   | Yes      | Watcher engagement signal                                                  |
| statuses_url      | Yes      | Commit status / CI outcome signals                                         |
| deployments_url   | Yes      | Deployment operations and engineering maturity                             |
| archive_url       | No       | Archive download generation endpoint                                       |
| assignees_url     | No       | Assignable users list (limited health signal value)                        |
| blobs_url         | No       | Low-level Git object access                                                |
| collaborators_url | No       | Permission-level collaborator listing                                      |
| compare_url       | No       | Commit comparison utility endpoint                                         |
| contents_url      | No       | Repository file browsing endpoint                                          |
| downloads_url     | No       | Legacy download artifacts endpoint                                         |
| git_commits_url   | No       | Low-level Git commit object endpoint                                       |
| git_refs_url      | No       | Low-level Git reference endpoint                                           |
| git_tags_url      | No       | Low-level Git tag object endpoint                                          |
| hooks_url         | No       | Webhook configuration endpoint                                             |
| keys_url          | No       | Deploy key management endpoint                                             |
| merges_url        | No       | Merge utility endpoint                                                     |
| notifications_url | No       | Notification subscription endpoint                                         |
| subscription_url  | No       | User subscription management endpoint                                      |
| teams_url         | No       | Organization team access endpoint                                          |
| trees_url         | No       | Low-level Git tree object endpoint                                         |


In [26]:
SELECTED_ENDPOINTS = {
    "repo": f"{BASE_URL}/repos/{FULL_NAME}",
    "contributors": f"{BASE_URL}/repos/{FULL_NAME}/contributors",
    "commits": f"{BASE_URL}/repos/{FULL_NAME}/commits",
    "issues": f"{BASE_URL}/repos/{FULL_NAME}/issues",
    "pulls": f"{BASE_URL}/repos/{FULL_NAME}/pulls",
    "comments": f"{BASE_URL}/repos/{FULL_NAME}/comments",
    "issue_comments": f"{BASE_URL}/repos/{FULL_NAME}/issues/comments",
    "issue_events": f"{BASE_URL}/repos/{FULL_NAME}/issues/events",
    "releases": f"{BASE_URL}/repos/{FULL_NAME}/releases",
    "tags": f"{BASE_URL}/repos/{FULL_NAME}/tags",
    "branches": f"{BASE_URL}/repos/{FULL_NAME}/branches",
    "languages": f"{BASE_URL}/repos/{FULL_NAME}/languages",
    "labels": f"{BASE_URL}/repos/{FULL_NAME}/labels",
    "milestones": f"{BASE_URL}/repos/{FULL_NAME}/milestones",
    "events": f"{BASE_URL}/repos/{FULL_NAME}/events",
    "forks": f"{BASE_URL}/repos/{FULL_NAME}/forks",
    "stargazers": f"{BASE_URL}/repos/{FULL_NAME}/stargazers",
    "subscribers": f"{BASE_URL}/repos/{FULL_NAME}/subscribers",
    "statuses": f"{BASE_URL}/repos/{FULL_NAME}/statuses/{{sha}}",  # needs commit sha
    "deployments": f"{BASE_URL}/repos/{FULL_NAME}/deployments",
}

## **[2nd Selection]**

In [36]:
# endpoint request params

ENDPOINT_PARAMS = {
    "repo": None,
    "contributors": {"per_page": 100},
    "commits": {"per_page": 100},
    "issues": {"state": "all", "per_page": 100},
    "pulls": {"state": "all", "per_page": 100},
    "comments": {"per_page": 100},
    "issue_comments": {"per_page": 100},
    "issue_events": {"per_page": 100},
    "releases": {"per_page": 100},
    "tags": {"per_page": 100},
    "branches": {"per_page": 100},
    "languages": None,
    "labels": {"per_page": 100},
    "milestones": {"state": "all", "per_page": 100},
    "events": {"per_page": 100},
    "forks": {"per_page": 100},
    "stargazers": {"per_page": 100},
    "subscribers": {"per_page": 100},
    "deployments": {"per_page": 100},
}

### [Section 1] helper: safe api call

In [27]:
def fetch_json(url, params=None, headers=HEADERS, sleep_sec=0.2):
    res = requests.get(url, headers=headers, params=params)
    res.raise_for_status()
    time.sleep(sleep_sec)
    return res.json()


def safe_fetch_json(url, params=None, headers=HEADERS, sleep_sec=0.2):
    try:
        return fetch_json(url, params=params, headers=headers, sleep_sec=sleep_sec)
    except Exception as e:
        print(f"[ERROR] {url} -> {e}")
        return None

### [Section 2] helper: value type / example shortening

In [28]:
def get_value_type(value):
    if value is None:
        return "NoneType"
    elif isinstance(value, bool):
        return "bool"
    elif isinstance(value, int):
        return "int"
    elif isinstance(value, float):
        return "float"
    elif isinstance(value, str):
        return "str"
    elif isinstance(value, dict):
        return "dict"
    elif isinstance(value, list):
        return "list"
    else:
        return type(value).__name__


def shorten_value(value, max_len=120):
    text = str(value)
    if len(text) > max_len:
        return text[:max_len] + "..."
    return text

### [Section 3] recursive walker
- endpoint response 내부의 모든 key path 추출

In [30]:
def walk_json(obj, parent_path=""):
    rows = []

    if isinstance(obj, dict):
        for key, value in obj.items():
            current_path = f"{parent_path}.{key}" if parent_path else key

            rows.append({
                "path": current_path,
                "key": key,
                "dtype": get_value_type(value),
                "example_value": shorten_value(value),
            })

            if isinstance(value, (dict, list)):
                rows.extend(walk_json(value, current_path))

    elif isinstance(obj, list):
        for idx, item in enumerate(obj[:10]):   # list 내부는 앞 10개만 schema 샘플링
            current_path = f"{parent_path}[{idx}]" if parent_path else f"[{idx}]"

            rows.append({
                "path": current_path,
                "key": f"[{idx}]",
                "dtype": get_value_type(item),
                "example_value": shorten_value(item),
            })

            if isinstance(item, (dict, list)):
                rows.extend(walk_json(item, current_path))

    return rows

### [Section 4] list endpoint의 union schema 정리
- list 응답이면 item 여러 개를 합쳐 key inventory 생성

In [31]:
def build_schema_inventory(endpoint_name, data):
    rows = []

    if data is None:
        return pd.DataFrame(columns=[
            "endpoint", "path", "key", "dtype", "example_value"
        ])

    if isinstance(data, dict):
        walked = walk_json(data)
        for row in walked:
            row["endpoint"] = endpoint_name
            rows.append(row)

    elif isinstance(data, list):
        seen = {}

        for item in data:
            walked = walk_json(item)

            for row in walked:
                path = row["path"]

                if path not in seen:
                    seen[path] = {
                        "endpoint": endpoint_name,
                        "path": row["path"],
                        "key": row["key"],
                        "dtype": set([row["dtype"]]),
                        "example_value": row["example_value"],
                    }
                else:
                    seen[path]["dtype"].add(row["dtype"])

        for path, meta in seen.items():
            rows.append({
                "endpoint": meta["endpoint"],
                "path": meta["path"],
                "key": meta["key"],
                "dtype": ", ".join(sorted(meta["dtype"])),
                "example_value": meta["example_value"],
            })

    df = pd.DataFrame(rows)

    if not df.empty:
        df = df.sort_values("path").reset_index(drop=True)

    return df

### [Section 5] statuses endpoint substitute sha
- statuses는 sha가 필요하므로 commits endpoint에서 하나 가져옴

In [33]:
sample_commit_sha = None

commits_data_for_sha = safe_fetch_json(
    SELECTED_ENDPOINTS["commits"],
    params={"per_page": 1}
)

if isinstance(commits_data_for_sha, list) and len(commits_data_for_sha) > 0:
    sample_commit_sha = commits_data_for_sha[0].get("sha")

print("sample_commit_sha:", sample_commit_sha)

sample_commit_sha: 2b4bfccb1032505533c571b112c5ab67681d9140


### [Section 6] final endpoint url map for schema extraction
- statuses는 sample sha로 치환

In [34]:
SCHEMA_ENDPOINTS = SELECTED_ENDPOINTS.copy()

if sample_commit_sha is not None:
    SCHEMA_ENDPOINTS["statuses"] = SELECTED_ENDPOINTS["statuses"].format(sha=sample_commit_sha)
else:
    SCHEMA_ENDPOINTS.pop("statuses", None)
    print("[INFO] statuses endpoint skipped because no sample sha was found.")

### [Section 7] fetch all selected endpoints

In [37]:
endpoint_response_map = {}

for endpoint_name, endpoint_url in SCHEMA_ENDPOINTS.items():
    print(f"Fetching: {endpoint_name}")
    data = safe_fetch_json(
        endpoint_url,
        params=ENDPOINT_PARAMS.get(endpoint_name)
    )
    endpoint_response_map[endpoint_name] = data

Fetching: repo
Fetching: contributors
Fetching: commits
Fetching: issues
Fetching: pulls
Fetching: comments
Fetching: issue_comments
Fetching: issue_events
Fetching: releases
Fetching: tags
Fetching: branches
Fetching: languages
Fetching: labels
Fetching: milestones
Fetching: events
Fetching: forks
Fetching: stargazers
Fetching: subscribers
Fetching: statuses
Fetching: deployments


### [Section 8] schema inventory for each endpoint

In [38]:
schema_frames = []

for endpoint_name, data in endpoint_response_map.items():
    df_schema = build_schema_inventory(endpoint_name, data)
    schema_frames.append(df_schema)

all_feature_inventory_df = pd.concat(schema_frames, ignore_index=True)

all_feature_inventory_df = all_feature_inventory_df.sort_values(
    ["endpoint", "path"]
).reset_index(drop=True)

print(all_feature_inventory_df.shape)
display(all_feature_inventory_df.head(100))

(2391, 5)


,path,key,dtype,example_value,endpoint
0,commit,commit,dict,{'sha': '12b3264d17780b6b38f747353bc80cfd910e6...,branches
1,commit.sha,sha,str,12b3264d17780b6b38f747353bc80cfd910e6dc8,branches
2,commit.url,url,str,https://api.github.com/repos/pandas-dev/pandas...,branches
3,name,name,str,0.19.x,branches
4,protected,protected,bool,True,branches
...,...,...,...,...,...
95,committer.gists_url,gists_url,str,https://api.github.com/users/web-flow/gists{/g...,commits
96,committer.gravatar_id,gravatar_id,str,,commits
97,committer.html_url,html_url,str,https://github.com/web-flow,commits
98,committer.id,id,int,19864447,commits


### [Section 9] endpoint-wise preview

In [41]:
for endpoint_name in all_feature_inventory_df["endpoint"].unique():
    print("\n" + "-" * 80)
    print(f"[Endpoint] {endpoint_name}")
    print("-" * 80)

    display(
        all_feature_inventory_df[
            all_feature_inventory_df["endpoint"] == endpoint_name
        ].reset_index(drop=True)
    )


--------------------------------------------------------------------------------
[Endpoint] branches
--------------------------------------------------------------------------------


,path,key,dtype,example_value,endpoint
0,commit,commit,dict,{'sha': '12b3264d17780b6b38f747353bc80cfd910e6...,branches
1,commit.sha,sha,str,12b3264d17780b6b38f747353bc80cfd910e6dc8,branches
2,commit.url,url,str,https://api.github.com/repos/pandas-dev/pandas...,branches
3,name,name,str,0.19.x,branches
4,protected,protected,bool,True,branches



--------------------------------------------------------------------------------
[Endpoint] comments
--------------------------------------------------------------------------------


,path,key,dtype,example_value,endpoint
0,author_association,author_association,str,CONTRIBUTOR,comments
1,body,body,str,Does this need to be done for series as well?\n,comments
2,commit_id,commit_id,str,ae4e51092d27520d3ed1ac1dcae658be33ff096a,comments
3,created_at,created_at,str,2011-10-09T20:23:35Z,comments
4,html_url,html_url,str,https://github.com/pandas-dev/pandas/commit/ae...,comments
5,id,id,int,640090,comments
6,line,line,"NoneType, int",501,comments
7,node_id,node_id,str,MDEzOkNvbW1pdENvbW1lbnQ2NDAwOTA=,comments
8,path,path,"NoneType, str",pandas/core/frame.py,comments
9,position,position,"NoneType, int",5,comments



--------------------------------------------------------------------------------
[Endpoint] commits
--------------------------------------------------------------------------------


...

,path,key,dtype,example_value,endpoint
0,avatar_url,avatar_url,str,https://avatars.githubusercontent.com/in/68672...,statuses
1,context,context,str,pre-commit.ci - push,statuses
2,created_at,created_at,str,2026-03-23T01:54:35Z,statuses
3,creator,creator,dict,"{'login': 'pre-commit-ci[bot]', 'id': 66853113...",statuses
4,creator.avatar_url,avatar_url,str,https://avatars.githubusercontent.com/in/68672...,statuses
5,creator.events_url,events_url,str,https://api.github.com/users/pre-commit-ci%5Bb...,statuses
6,creator.followers_url,followers_url,str,https://api.github.com/users/pre-commit-ci%5Bb...,statuses
7,creator.following_url,following_url,str,https://api.github.com/users/pre-commit-ci%5Bb...,statuses
8,creator.gists_url,gists_url,str,https://api.github.com/users/pre-commit-ci%5Bb...,statuses
9,creator.gravatar_id,gravatar_id,str,,statuses



--------------------------------------------------------------------------------
[Endpoint] subscribers
--------------------------------------------------------------------------------


,path,key,dtype,example_value,endpoint
0,avatar_url,avatar_url,str,https://avatars.githubusercontent.com/u/759245...,subscribers
1,events_url,events_url,str,https://api.github.com/users/changhiskhan/even...,subscribers
2,followers_url,followers_url,str,https://api.github.com/users/changhiskhan/foll...,subscribers
3,following_url,following_url,str,https://api.github.com/users/changhiskhan/foll...,subscribers
4,gists_url,gists_url,str,https://api.github.com/users/changhiskhan/gist...,subscribers
5,gravatar_id,gravatar_id,str,,subscribers
6,html_url,html_url,str,https://github.com/changhiskhan,subscribers
7,id,id,int,759245,subscribers
8,login,login,str,changhiskhan,subscribers
9,node_id,node_id,str,MDQ6VXNlcjc1OTI0NQ==,subscribers



--------------------------------------------------------------------------------
[Endpoint] tags
--------------------------------------------------------------------------------


,path,key,dtype,example_value,endpoint
0,commit,commit,dict,{'sha': '1797ba54796ec1dc726cb6dd5039c238caec3...,tags
1,commit.sha,sha,str,1797ba54796ec1dc726cb6dd5039c238caec30a2,tags
2,commit.url,url,str,https://api.github.com/repos/pandas-dev/pandas...,tags
3,name,name,str,v3.1.0.dev0,tags
4,node_id,node_id,str,MDM6UmVmODU4MTI3OnJlZnMvdGFncy92My4xLjAuZGV2MA==,tags
5,tarball_url,tarball_url,str,https://api.github.com/repos/pandas-dev/pandas...,tags
6,zipball_url,zipball_url,str,https://api.github.com/repos/pandas-dev/pandas...,tags


In [42]:
all_feature_inventory_df.to_csv("output/endpoint_feature_inventory.csv", index=False)
print("Saved: endpoint_feature_inventory.csv")

Saved: endpoint_feature_inventory.csv


### **[2nd Selection — Endpoint Mapping based on Health Indicators]**

### Community Activity

| Indicator          | Selected Endpoint | Description                                                                         |
| ------------------ | ----------------- | ----------------------------------------------------------------------------------- |
| Activity Volume    | `commits`         | Provides commit history to measure contribution volume and temporal activity trends |
| Activity Volume    | `issues`          | Tracks issue creation flow reflecting project interaction intensity                 |
| Activity Volume    | `pulls`           | Captures pull request activity indicating contribution throughput                   |
| Activity Volume    | `contributors`    | Shows contributor participation level and contribution counts                       |
| Responsiveness     | `issue_events`    | Contains issue lifecycle events enabling response time analysis                     |
| Responsiveness     | `issue_comments`  | Provides timestamps of discussion responses on issues                               |
| Engagement Quality | `comments`        | Reflects discussion density and conversational participation patterns               |
| Engagement Quality | `events`          | Captures various repository interaction events indicating community engagement      |

---

### Sustainability

| Indicator             | Selected Endpoint | Description                                                              |
| --------------------- | ----------------- | ------------------------------------------------------------------------ |
| Contributor Structure | `contributors`    | Enables analysis of contribution concentration and bus-factor risk       |
| Diversity             | `contributors`    | Supports measurement of contributor diversity and new participant inflow |
| Activity Stability    | `commits`         | Allows time-series analysis of development continuity                    |
| Activity Stability    | `releases`        | Release timing helps evaluate long-term activity regularity              |

---

### Code Quality & Reliability

| Indicator            | Selected Endpoint | Description                                                                |
| -------------------- | ----------------- | -------------------------------------------------------------------------- |
| Engineering Practice | `branches`        | Branch protection and workflow structure reflect development maturity      |
| Engineering Practice | `pulls`           | PR lifecycle and review participation indicate process discipline          |
| Defect Signals       | `issues`          | Issue state transitions provide signals about defect management efficiency |
| Defect Signals       | `labels`          | Bug-related labels support classification of defect handling patterns      |
| Security Signals     | `deployments`     | Deployment records indicate operational robustness and release governance  |
| Security Signals     | `statuses`        | Commit status checks reveal CI/CD verification practices                   |

---

### Legal & Governance

| Indicator            | Selected Endpoint | Description                                                           |
| -------------------- | ----------------- | --------------------------------------------------------------------- |
| Legal Compliance     | `repo`            | Repository metadata includes license and visibility information       |
| Governance Structure | `labels`          | Label taxonomy reflects project management conventions                |
| Governance Structure | `milestones`      | Milestones show planning structure and coordinated development cycles |

---

### Project Maturity

| Indicator             | Selected Endpoint | Description                                                                    |
| --------------------- | ----------------- | ------------------------------------------------------------------------------ |
| Release Engineering   | `releases`        | Release history reveals versioning discipline and delivery cadence             |
| Release Engineering   | `tags`            | Tag creation patterns provide lightweight signals of version control practices |
| Adoption / Popularity | `stargazers`      | Star activity reflects external recognition and ecosystem adoption             |
| Adoption / Popularity | `forks`           | Fork events indicate downstream usage and community interest                   |
| Adoption / Popularity | `subscribers`     | Watcher behavior shows sustained interest in project evolution                 |
| Lifecycle / Scale     | `repo`            | Core repository metrics (size, age, issue volume) indicate maturity scale      |

---

## **[3rd Selection]**

### [Section 1] Indicator-Endpoint Map

In [43]:
INDICATOR_ENDPOINT_MAP = {

    # Community Activity
    "community_activity": [
        "commits",
        "issues",
        "pulls",
        "contributors",
        "issue_events",
        "issue_comments",
        "comments",
        "events",
    ],

    # Sustainability
    "sustainability": [
        "contributors",
        "commits",
        "releases",
    ],

    # Code Quality & Reliability
    "code_quality_reliability": [
        "branches",
        "pulls",
        "issues",
        "labels",
        "deployments",
        "statuses",
    ],

    # Legal & Governance
    "legal_governance": [
        "repo",
        "labels",
        "milestones",
    ],

    # Project Maturity
    "project_maturity": [
        "releases",
        "tags",
        "stargazers",
        "forks",
        "subscribers",
        "repo",
    ],
}

### [Section 2] Unique endpoint list from indicator map

In [45]:
selected_endpoint_list = sorted(
    set(
        endpoint
        for endpoint_list in INDICATOR_ENDPOINT_MAP.values()
        for endpoint in endpoint_list
    )
)

print("Total selected endpoints:", len(selected_endpoint_list))
print(selected_endpoint_list)

Total selected endpoints: 19
['branches', 'comments', 'commits', 'contributors', 'deployments', 'events', 'forks', 'issue_comments', 'issue_events', 'issues', 'labels', 'milestones', 'pulls', 'releases', 'repo', 'stargazers', 'statuses', 'subscribers', 'tags']


### [Section 3] Endpoint URL map

In [46]:
SELECTED_ENDPOINTS = {
    "repo": f"{BASE_URL}/repos/{FULL_NAME}",
    "contributors": f"{BASE_URL}/repos/{FULL_NAME}/contributors",
    "commits": f"{BASE_URL}/repos/{FULL_NAME}/commits",
    "issues": f"{BASE_URL}/repos/{FULL_NAME}/issues",
    "pulls": f"{BASE_URL}/repos/{FULL_NAME}/pulls",
    "comments": f"{BASE_URL}/repos/{FULL_NAME}/comments",
    "issue_comments": f"{BASE_URL}/repos/{FULL_NAME}/issues/comments",
    "issue_events": f"{BASE_URL}/repos/{FULL_NAME}/issues/events",
    "releases": f"{BASE_URL}/repos/{FULL_NAME}/releases",
    "tags": f"{BASE_URL}/repos/{FULL_NAME}/tags",
    "branches": f"{BASE_URL}/repos/{FULL_NAME}/branches",
    "labels": f"{BASE_URL}/repos/{FULL_NAME}/labels",
    "milestones": f"{BASE_URL}/repos/{FULL_NAME}/milestones",
    "events": f"{BASE_URL}/repos/{FULL_NAME}/events",
    "forks": f"{BASE_URL}/repos/{FULL_NAME}/forks",
    "stargazers": f"{BASE_URL}/repos/{FULL_NAME}/stargazers",
    "subscribers": f"{BASE_URL}/repos/{FULL_NAME}/subscribers",
    "statuses": f"{BASE_URL}/repos/{FULL_NAME}/statuses/{{sha}}",   # needs sha
    "deployments": f"{BASE_URL}/repos/{FULL_NAME}/deployments",
}

### [Section 4] Request params for each endpoint

In [ ]:
ENDPOINT_PARAMS = {
    "repo": None,
    "contributors": {"per_page": 100},
    "commits": {"per_page": 100},
    "issues": {"state": "all", "per_page": 100},
    "pulls": {"state": "all", "per_page": 100},
    "comments": {"per_page": 100},
    "issue_comments": {"per_page": 100},
    "issue_events": {"per_page": 100},
    "releases": {"per_page": 100},
    "tags": {"per_page": 100},
    "branches": {"per_page": 100},
    "labels": {"per_page": 100},
    "milestones": {"state": "all", "per_page": 100},
    "events": {"per_page": 100},
    "forks": {"per_page": 100},
    "stargazers": {"per_page": 100},
    "subscribers": {"per_page": 100},
    "deployments": {"per_page": 100},
}

### [Section 5] Safe fetch helpers

In [50]:
def fetch_json(url, params=None, headers=HEADERS, sleep_sec=0.2):
    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()
    time.sleep(sleep_sec)
    return response.json()


def safe_fetch_json(url, params=None, headers=HEADERS, sleep_sec=0.2):
    try:
        return fetch_json(url, params=params, headers=headers, sleep_sec=sleep_sec)
    except Exception as e:
        print(f"[ERROR] {url} -> {e}")
        return None

### [Section 6] Value type / example helpers

In [51]:
def get_value_type(value):
    if value is None:
        return "NoneType"
    elif isinstance(value, bool):
        return "bool"
    elif isinstance(value, int):
        return "int"
    elif isinstance(value, float):
        return "float"
    elif isinstance(value, str):
        return "str"
    elif isinstance(value, dict):
        return "dict"
    elif isinstance(value, list):
        return "list"
    else:
        return type(value).__name__


def shorten_value(value, max_len=120):
    text = str(value)
    if len(text) > max_len:
        return text[:max_len] + "..."
    return text

### [Section 7] Recursive JSON walker
- extracts every possible key path inside endpoint response


In [52]:
def walk_json(obj, parent_path=""):
    rows = []

    if isinstance(obj, dict):
        for key, value in obj.items():
            current_path = f"{parent_path}.{key}" if parent_path else key

            rows.append({
                "path": current_path,
                "key": key,
                "dtype": get_value_type(value),
                "example_value": shorten_value(value),
            })

            if isinstance(value, (dict, list)):
                rows.extend(walk_json(value, current_path))

    elif isinstance(obj, list):
        for idx, item in enumerate(obj[:10]):   # schema sampling for list items
            current_path = f"{parent_path}[{idx}]" if parent_path else f"[{idx}]"

            rows.append({
                "path": current_path,
                "key": f"[{idx}]",
                "dtype": get_value_type(item),
                "example_value": shorten_value(item),
            })

            if isinstance(item, (dict, list)):
                rows.extend(walk_json(item, current_path))

    return rows

### [Section 8] Build endpoint schema inventory
- for list responses, union all discovered paths across items

In [53]:
def build_schema_inventory(endpoint_name, data):
    rows = []

    if data is None:
        return pd.DataFrame(columns=[
            "endpoint", "path", "key", "dtype", "example_value"
        ])

    if isinstance(data, dict):
        walked = walk_json(data)
        for row in walked:
            row["endpoint"] = endpoint_name
            rows.append(row)

    elif isinstance(data, list):
        seen = {}

        for item in data:
            walked = walk_json(item)

            for row in walked:
                path = row["path"]

                if path not in seen:
                    seen[path] = {
                        "endpoint": endpoint_name,
                        "path": row["path"],
                        "key": row["key"],
                        "dtype_set": set([row["dtype"]]),
                        "example_value": row["example_value"],
                    }
                else:
                    seen[path]["dtype_set"].add(row["dtype"])

        for path, meta in seen.items():
            rows.append({
                "endpoint": meta["endpoint"],
                "path": meta["path"],
                "key": meta["key"],
                "dtype": ", ".join(sorted(meta["dtype_set"])),
                "example_value": meta["example_value"],
            })

    df = pd.DataFrame(rows)

    if not df.empty:
        df = df.sort_values("path").reset_index(drop=True)

    return df

### [Section 9] Resolve statuses endpoint with a sample commit sha

In [54]:
sample_commit_sha = None

commits_preview = safe_fetch_json(
    SELECTED_ENDPOINTS["commits"],
    params={"per_page": 1}
)

if isinstance(commits_preview, list) and len(commits_preview) > 0:
    sample_commit_sha = commits_preview[0].get("sha")

print("sample_commit_sha:", sample_commit_sha)

sample_commit_sha: 2b4bfccb1032505533c571b112c5ab67681d9140


### [Section 10] Final endpoint map for 3rd selection
- uses only endpoints referenced in INDICATOR_ENDPOINT_MAP


In [56]:
THIRD_SELECTION_ENDPOINTS = {}

for endpoint_name in selected_endpoint_list:
    if endpoint_name == "statuses":
        if sample_commit_sha is not None:
            THIRD_SELECTION_ENDPOINTS[endpoint_name] = SELECTED_ENDPOINTS["statuses"].format(
                sha=sample_commit_sha
            )
        else:
            print("[INFO] statuses skipped because sample sha was not found.")
    else:
        THIRD_SELECTION_ENDPOINTS[endpoint_name] = SELECTED_ENDPOINTS[endpoint_name]

print("Total endpoints for 3rd selection:", len(THIRD_SELECTION_ENDPOINTS))
print(list(THIRD_SELECTION_ENDPOINTS.keys()))

Total endpoints for 3rd selection: 19
['branches', 'comments', 'commits', 'contributors', 'deployments', 'events', 'forks', 'issue_comments', 'issue_events', 'issues', 'labels', 'milestones', 'pulls', 'releases', 'repo', 'stargazers', 'statuses', 'subscribers', 'tags']


### [Section 11] Fetch endpoint responses

In [57]:
endpoint_response_map = {}

for endpoint_name, endpoint_url in THIRD_SELECTION_ENDPOINTS.items():
    print(f"Fetching: {endpoint_name}")
    endpoint_response_map[endpoint_name] = safe_fetch_json(
        endpoint_url,
        params=ENDPOINT_PARAMS.get(endpoint_name)
    )

Fetching: branches
Fetching: comments
Fetching: commits
Fetching: contributors
Fetching: deployments
Fetching: events
Fetching: forks
Fetching: issue_comments
Fetching: issue_events
Fetching: issues
Fetching: labels
Fetching: milestones
Fetching: pulls
Fetching: releases
Fetching: repo
Fetching: stargazers
Fetching: statuses
Fetching: subscribers
Fetching: tags


### [Section 12] Build full key inventory across endpoints

In [58]:
schema_frames = []

for endpoint_name, data in endpoint_response_map.items():
    df_schema = build_schema_inventory(endpoint_name, data)
    schema_frames.append(df_schema)

all_key_inventory_df = pd.concat(schema_frames, ignore_index=True)

all_key_inventory_df = all_key_inventory_df.sort_values(
    ["endpoint", "path"]
).reset_index(drop=True)

print(all_key_inventory_df.shape)
display(all_key_inventory_df.head(100))

(2380, 5)


,endpoint,path,key,dtype,example_value
0,branches,commit,commit,dict,{'sha': '12b3264d17780b6b38f747353bc80cfd910e6...
1,branches,commit.sha,sha,str,12b3264d17780b6b38f747353bc80cfd910e6dc8
2,branches,commit.url,url,str,https://api.github.com/repos/pandas-dev/pandas...
3,branches,name,name,str,0.19.x
4,branches,protected,protected,bool,True
...,...,...,...,...,...
95,commits,committer.gists_url,gists_url,str,https://api.github.com/users/web-flow/gists{/g...
96,commits,committer.gravatar_id,gravatar_id,str,
97,commits,committer.html_url,html_url,str,https://github.com/web-flow
98,commits,committer.id,id,int,19864447


### [Section 13] Add indicator linkage
- shows which indicator(s) each endpoint belongs to

In [59]:
endpoint_to_indicators = {}

for indicator_name, endpoint_names in INDICATOR_ENDPOINT_MAP.items():
    for endpoint_name in endpoint_names:
        endpoint_to_indicators.setdefault(endpoint_name, []).append(indicator_name)

all_key_inventory_df["indicators"] = all_key_inventory_df["endpoint"].map(
    lambda x: ", ".join(endpoint_to_indicators.get(x, []))
)

### [Section 14] Reorder final inventory columns

In [60]:
all_key_inventory_df = all_key_inventory_df[
    ["indicators", "endpoint", "path", "key", "dtype", "example_value"]
]

display(all_key_inventory_df.head(100))

,indicators,endpoint,path,key,dtype,example_value
0,code_quality_reliability,branches,commit,commit,dict,{'sha': '12b3264d17780b6b38f747353bc80cfd910e6...
1,code_quality_reliability,branches,commit.sha,sha,str,12b3264d17780b6b38f747353bc80cfd910e6dc8
2,code_quality_reliability,branches,commit.url,url,str,https://api.github.com/repos/pandas-dev/pandas...
3,code_quality_reliability,branches,name,name,str,0.19.x
4,code_quality_reliability,branches,protected,protected,bool,True
...,...,...,...,...,...,...
95,"community_activity, sustainability",commits,committer.gists_url,gists_url,str,https://api.github.com/users/web-flow/gists{/g...
96,"community_activity, sustainability",commits,committer.gravatar_id,gravatar_id,str,
97,"community_activity, sustainability",commits,committer.html_url,html_url,str,https://github.com/web-flow
98,"community_activity, sustainability",commits,committer.id,id,int,19864447


### [Section 15] Preview endpoint-wise key inventory

In [62]:
for endpoint_name in all_key_inventory_df["endpoint"].unique():
    print("\n" + "-" * 80)
    print(f"[Endpoint] {endpoint_name}")
    print("-" * 80)

    display(
        all_key_inventory_df[
            all_key_inventory_df["endpoint"] == endpoint_name
        ].reset_index(drop=True)
    )


--------------------------------------------------------------------------------
[Endpoint] branches
--------------------------------------------------------------------------------


,indicators,endpoint,path,key,dtype,example_value
0,code_quality_reliability,branches,commit,commit,dict,{'sha': '12b3264d17780b6b38f747353bc80cfd910e6...
1,code_quality_reliability,branches,commit.sha,sha,str,12b3264d17780b6b38f747353bc80cfd910e6dc8
2,code_quality_reliability,branches,commit.url,url,str,https://api.github.com/repos/pandas-dev/pandas...
3,code_quality_reliability,branches,name,name,str,0.19.x
4,code_quality_reliability,branches,protected,protected,bool,True



--------------------------------------------------------------------------------
[Endpoint] comments
--------------------------------------------------------------------------------


,indicators,endpoint,path,key,dtype,example_value
0,community_activity,comments,author_association,author_association,str,CONTRIBUTOR
1,community_activity,comments,body,body,str,Does this need to be done for series as well?\n
2,community_activity,comments,commit_id,commit_id,str,ae4e51092d27520d3ed1ac1dcae658be33ff096a
3,community_activity,comments,created_at,created_at,str,2011-10-09T20:23:35Z
4,community_activity,comments,html_url,html_url,str,https://github.com/pandas-dev/pandas/commit/ae...
5,community_activity,comments,id,id,int,640090
6,community_activity,comments,line,line,"NoneType, int",501
7,community_activity,comments,node_id,node_id,str,MDEzOkNvbW1pdENvbW1lbnQ2NDAwOTA=
8,community_activity,comments,path,path,"NoneType, str",pandas/core/frame.py
9,community_activity,comments,position,position,"NoneType, int",5



--------------------------------------------------------------------------------
[Endpoint] commits
--------------------------------------------------------------------------------


...

,indicators,endpoint,path,key,dtype,example_value
0,code_quality_reliability,statuses,avatar_url,avatar_url,str,https://avatars.githubusercontent.com/in/68672...
1,code_quality_reliability,statuses,context,context,str,pre-commit.ci - push
2,code_quality_reliability,statuses,created_at,created_at,str,2026-03-23T01:54:35Z
3,code_quality_reliability,statuses,creator,creator,dict,"{'login': 'pre-commit-ci[bot]', 'id': 66853113..."
4,code_quality_reliability,statuses,creator.avatar_url,avatar_url,str,https://avatars.githubusercontent.com/in/68672...
5,code_quality_reliability,statuses,creator.events_url,events_url,str,https://api.github.com/users/pre-commit-ci%5Bb...
6,code_quality_reliability,statuses,creator.followers_url,followers_url,str,https://api.github.com/users/pre-commit-ci%5Bb...
7,code_quality_reliability,statuses,creator.following_url,following_url,str,https://api.github.com/users/pre-commit-ci%5Bb...
8,code_quality_reliability,statuses,creator.gists_url,gists_url,str,https://api.github.com/users/pre-commit-ci%5Bb...
9,code_quality_reliability,statuses,creator.gravatar_id,gravatar_id,str,



--------------------------------------------------------------------------------
[Endpoint] subscribers
--------------------------------------------------------------------------------


,indicators,endpoint,path,key,dtype,example_value
0,project_maturity,subscribers,avatar_url,avatar_url,str,https://avatars.githubusercontent.com/u/759245...
1,project_maturity,subscribers,events_url,events_url,str,https://api.github.com/users/changhiskhan/even...
2,project_maturity,subscribers,followers_url,followers_url,str,https://api.github.com/users/changhiskhan/foll...
3,project_maturity,subscribers,following_url,following_url,str,https://api.github.com/users/changhiskhan/foll...
4,project_maturity,subscribers,gists_url,gists_url,str,https://api.github.com/users/changhiskhan/gist...
5,project_maturity,subscribers,gravatar_id,gravatar_id,str,
6,project_maturity,subscribers,html_url,html_url,str,https://github.com/changhiskhan
7,project_maturity,subscribers,id,id,int,759245
8,project_maturity,subscribers,login,login,str,changhiskhan
9,project_maturity,subscribers,node_id,node_id,str,MDQ6VXNlcjc1OTI0NQ==



--------------------------------------------------------------------------------
[Endpoint] tags
--------------------------------------------------------------------------------


,indicators,endpoint,path,key,dtype,example_value
0,project_maturity,tags,commit,commit,dict,{'sha': '1797ba54796ec1dc726cb6dd5039c238caec3...
1,project_maturity,tags,commit.sha,sha,str,1797ba54796ec1dc726cb6dd5039c238caec30a2
2,project_maturity,tags,commit.url,url,str,https://api.github.com/repos/pandas-dev/pandas...
3,project_maturity,tags,name,name,str,v3.1.0.dev0
4,project_maturity,tags,node_id,node_id,str,MDM6UmVmODU4MTI3OnJlZnMvdGFncy92My4xLjAuZGV2MA==
5,project_maturity,tags,tarball_url,tarball_url,str,https://api.github.com/repos/pandas-dev/pandas...
6,project_maturity,tags,zipball_url,zipball_url,str,https://api.github.com/repos/pandas-dev/pandas...


### [Section 16] Save as csv file

In [63]:
all_key_inventory_df.to_csv("output/third_selection_key_inventory.csv", index=False)
print("Saved: third_selection_key_inventory.csv")

Saved: third_selection_key_inventory.csv


**3rd Selection — Raw Key Screening Logic**

- selected endpoints에서 모든 JSON key path 추출 → **raw key inventory 생성**

- 각 key를 아래 기준으로 logical screening 수행  
  - indicator relevance (지표와 개념적으로 연결되는가)  
  - measurability (수치화/범주화 가능한가)  
  - cross-repo comparability (다른 repo와 비교 가능한가)  
  - non-redundancy (중복 metadata 아닌가)  
  - noise risk (URL, node_id 등 API navigation field인가)

- rule-based decision 적용  
  - **drop**: noise 높고 relevance 없음  
  - **keep_candidate**: relevance 있고 measurable  
  - **review**: 애매 → human 판단 필요

- drop 제외한 key만 남김 
→ **final candidate set 구성 (human feature selection input)**

### [Section 17] 3rd Selection Rule Definitions

In [64]:
# Hard drop patterns (structural noise)
DROP_PATTERNS = [
    "_url",
    "avatar",
    "gravatar",
    "node_id",
    "href",
]

# Strong keep signal keywords
KEEP_PATTERNS = [
    "created_at",
    "updated_at",
    "closed_at",
    "merged_at",
    "published_at",
    "state",
    "protected",
    "draft",
    "locked",
    "comments",
    "comment_count",
    "contributions",
    "login",
    "license",
    "milestone",
    "tag",
    "environment",
    "verified",
    "fork",
    "star",
    "subscriber",
]

### [Section 18] Logical scoring function

In [65]:
def evaluate_key_logic(row):

    path = row["path"].lower()
    dtype = row["dtype"]

    relevance_score = 0
    measurable_score = 0
    noise_score = 0

    # Indicator relevance proxy
    for kw in KEEP_PATTERNS:
        if kw in path:
            relevance_score += 1

    # Measurability proxy
    if dtype in ["int", "float", "bool"]:
        measurable_score += 2
    elif dtype == "str":
        measurable_score += 1

    # Noise proxy
    for kw in DROP_PATTERNS:
        if kw in path:
            noise_score += 2

    return relevance_score, measurable_score, noise_score

### [Section 19] Apply scoring

In [66]:
scores = all_key_inventory_df.apply(
    lambda row: evaluate_key_logic(row),
    axis=1,
)

all_key_inventory_df["relevance_score"] = [s[0] for s in scores]
all_key_inventory_df["measurable_score"] = [s[1] for s in scores]
all_key_inventory_df["noise_score"] = [s[2] for s in scores]

### [Section 20] Decision rule (core pruning logic)

In [67]:
def decide_selection(row):

    if row["noise_score"] >= 2 and row["relevance_score"] == 0:
        return "drop"

    if (
        row["relevance_score"] >= 1
        and row["measurable_score"] >= 1
        and row["noise_score"] == 0
    ):
        return "keep_candidate"

    return "review"


all_key_inventory_df["decision"] = all_key_inventory_df.apply(
    decide_selection,
    axis=1,
)

### [Section 21] Pruning summary

In [68]:
print(all_key_inventory_df["decision"].value_counts())

decision
review            1259
drop               825
keep_candidate     296
Name: count, dtype: int64


### [Section 22] Build reduced human-review table

In [69]:
review_df = all_key_inventory_df[
    all_key_inventory_df["decision"] != "drop"
].copy()

review_df = review_df.sort_values(
    ["endpoint", "relevance_score", "noise_score"],
    ascending=[True, False, True],
)

print("Reduced rows for human review:", len(review_df))
display(review_df.head(100))

Reduced rows for human review: 1555


,indicators,endpoint,path,key,dtype,example_value,relevance_score,measurable_score,noise_score,decision
4,code_quality_reliability,branches,protected,protected,bool,True,1,2,0,keep_candidate
0,code_quality_reliability,branches,commit,commit,dict,{'sha': '12b3264d17780b6b38f747353bc80cfd910e6...,0,0,0,review
1,code_quality_reliability,branches,commit.sha,sha,str,12b3264d17780b6b38f747353bc80cfd910e6dc8,0,1,0,review
2,code_quality_reliability,branches,commit.url,url,str,https://api.github.com/repos/pandas-dev/pandas...,0,1,0,review
3,code_quality_reliability,branches,name,name,str,0.19.x,0,1,0,review
...,...,...,...,...,...,...,...,...,...,...
237,code_quality_reliability,deployments,updated_at,updated_at,str,2026-02-17T22:20:21Z,1,1,0,keep_candidate
155,code_quality_reliability,deployments,creator.starred_url,starred_url,str,https://api.github.com/users/jorisvandenbossch...,1,1,2,review
201,code_quality_reliability,deployments,performed_via_github_app.owner.starred_url,starred_url,str,https://api.github.com/users/github/starred{/o...,1,1,2,review
140,code_quality_reliability,deployments,creator,creator,dict,"{'login': 'jorisvandenbossche', 'id': 1020496,...",0,0,0,review


### [Section 23] Endpoint-wise candidate preview

In [72]:
for ep in review_df["endpoint"].unique():

    print("\n" + "-"*70)
    print("Endpoint:", ep)
    print("-"*70)

    display(
        review_df[
            review_df["endpoint"] == ep
        ].reset_index(drop=True)
    )


----------------------------------------------------------------------
Endpoint: branches
----------------------------------------------------------------------


,indicators,endpoint,path,key,dtype,example_value,relevance_score,measurable_score,noise_score,decision
0,code_quality_reliability,branches,protected,protected,bool,True,1,2,0,keep_candidate
1,code_quality_reliability,branches,commit,commit,dict,{'sha': '12b3264d17780b6b38f747353bc80cfd910e6...,0,0,0,review
2,code_quality_reliability,branches,commit.sha,sha,str,12b3264d17780b6b38f747353bc80cfd910e6dc8,0,1,0,review
3,code_quality_reliability,branches,commit.url,url,str,https://api.github.com/repos/pandas-dev/pandas...,0,1,0,review
4,code_quality_reliability,branches,name,name,str,0.19.x,0,1,0,review



----------------------------------------------------------------------
Endpoint: comments
----------------------------------------------------------------------


,indicators,endpoint,path,key,dtype,example_value,relevance_score,measurable_score,noise_score,decision
0,community_activity,comments,created_at,created_at,str,2011-10-09T20:23:35Z,1,1,0,keep_candidate
1,community_activity,comments,updated_at,updated_at,str,2011-10-09T20:23:35Z,1,1,0,keep_candidate
2,community_activity,comments,user.login,login,str,takluyver,1,1,0,keep_candidate
3,community_activity,comments,user.starred_url,starred_url,str,https://api.github.com/users/takluyver/starred...,1,1,2,review
4,community_activity,comments,author_association,author_association,str,CONTRIBUTOR,0,1,0,review
5,community_activity,comments,body,body,str,Does this need to be done for series as well?\n,0,1,0,review
6,community_activity,comments,commit_id,commit_id,str,ae4e51092d27520d3ed1ac1dcae658be33ff096a,0,1,0,review
7,community_activity,comments,id,id,int,640090,0,2,0,review
8,community_activity,comments,line,line,"NoneType, int",501,0,0,0,review
9,community_activity,comments,path,path,"NoneType, str",pandas/core/frame.py,0,0,0,review



----------------------------------------------------------------------
Endpoint: commits
----------------------------------------------------------------------


...

,indicators,endpoint,path,key,dtype,example_value,relevance_score,measurable_score,noise_score,decision
0,code_quality_reliability,statuses,created_at,created_at,str,2026-03-23T01:54:35Z,1,1,0,keep_candidate
1,code_quality_reliability,statuses,creator.login,login,str,pre-commit-ci[bot],1,1,0,keep_candidate
2,code_quality_reliability,statuses,state,state,str,success,1,1,0,keep_candidate
3,code_quality_reliability,statuses,updated_at,updated_at,str,2026-03-23T01:54:35Z,1,1,0,keep_candidate
4,code_quality_reliability,statuses,creator.starred_url,starred_url,str,https://api.github.com/users/pre-commit-ci%5Bb...,1,1,2,review
5,code_quality_reliability,statuses,context,context,str,pre-commit.ci - push,0,1,0,review
6,code_quality_reliability,statuses,creator,creator,dict,"{'login': 'pre-commit-ci[bot]', 'id': 66853113...",0,0,0,review
7,code_quality_reliability,statuses,creator.id,id,int,66853113,0,2,0,review
8,code_quality_reliability,statuses,creator.site_admin,site_admin,bool,False,0,2,0,review
9,code_quality_reliability,statuses,creator.type,type,str,Bot,0,1,0,review



----------------------------------------------------------------------
Endpoint: subscribers
----------------------------------------------------------------------


,indicators,endpoint,path,key,dtype,example_value,relevance_score,measurable_score,noise_score,decision
0,project_maturity,subscribers,login,login,str,changhiskhan,1,1,0,keep_candidate
1,project_maturity,subscribers,starred_url,starred_url,str,https://api.github.com/users/changhiskhan/star...,1,1,2,review
2,project_maturity,subscribers,id,id,int,759245,0,2,0,review
3,project_maturity,subscribers,site_admin,site_admin,bool,False,0,2,0,review
4,project_maturity,subscribers,type,type,str,User,0,1,0,review
5,project_maturity,subscribers,url,url,str,https://api.github.com/users/changhiskhan,0,1,0,review
6,project_maturity,subscribers,user_view_type,user_view_type,str,public,0,1,0,review



----------------------------------------------------------------------
Endpoint: tags
----------------------------------------------------------------------


,indicators,endpoint,path,key,dtype,example_value,relevance_score,measurable_score,noise_score,decision
0,project_maturity,tags,commit,commit,dict,{'sha': '1797ba54796ec1dc726cb6dd5039c238caec3...,0,0,0,review
1,project_maturity,tags,commit.sha,sha,str,1797ba54796ec1dc726cb6dd5039c238caec30a2,0,1,0,review
2,project_maturity,tags,commit.url,url,str,https://api.github.com/repos/pandas-dev/pandas...,0,1,0,review
3,project_maturity,tags,name,name,str,v3.1.0.dev0,0,1,0,review


### [Section 24] Save pruning result

In [ ]:
review_df.to_csv("output/third_selection_candidates.csv", index=False)
print("Saved: third_selection_candidates.csv")

Saved: third_selection_candidates.csv


### [Section 25] Strong hard-drop rule for non-analytical fields

In [79]:
STRONG_DROP_PATTERNS = [
    "_url",
    "url",
    "avatar",
    "gravatar",
    "node_id",
    "href",
    "html",
    "following",
    "followers",
    "gists",
    "starred",
    "subscriptions",
    "organizations",
    "received_events",
    "repos_url",
]

STRONG_DROP_EXACT_KEYS = {
    "url",
    "html_url",
    "avatar_url",
    "gravatar_id",
    "node_id",
    "followers_url",
    "following_url",
    "gists_url",
    "starred_url",
    "subscriptions_url",
    "organizations_url",
    "received_events_url",
    "repos_url",
}

### [Section 26] Strong drop classifier

In [80]:
def is_strong_drop_field(path, key):
    path_lower = str(path).lower()
    key_lower = str(key).lower()

    if key_lower in STRONG_DROP_EXACT_KEYS:
        return True

    for pattern in STRONG_DROP_PATTERNS:
        if pattern in path_lower:
            return True

    return False

### [Section 27] Apply strong pruning

In [81]:
review_df["strong_drop"] = review_df.apply(
    lambda row: is_strong_drop_field(row["path"], row["key"]),
    axis=1
)

### [Section 28] Split after strong pruning

In [82]:
review_df_after_strong = review_df[review_df["strong_drop"] == False].copy()
dropped_by_strong_rule_df = review_df[review_df["strong_drop"] == True].copy()

print("Before strong pruning:", len(review_df))
print("After strong pruning:", len(review_df_after_strong))
print("Dropped by strong rule:", len(dropped_by_strong_rule_df))

Before strong pruning: 1555
After strong pruning: 1215
Dropped by strong rule: 340


### [Section 29] Preview dropped rows

In [83]:
display(
    dropped_by_strong_rule_df[
        ["indicators", "endpoint", "path", "key", "dtype", "decision"]
    ].head(100)
)

,indicators,endpoint,path,key,dtype,decision
2,code_quality_reliability,branches,commit.url,url,str,review
43,community_activity,comments,user.starred_url,starred_url,str,review
25,community_activity,comments,reactions.url,url,str,review
27,community_activity,comments,url,url,str,review
46,community_activity,comments,user.url,url,str,review
...,...,...,...,...,...,...
562,community_activity,events,payload.issue.milestone.creator.repos_url,repos_url,str,review
565,community_activity,events,payload.issue.milestone.creator.subscriptions_url,subscriptions_url,str,review
571,community_activity,events,payload.issue.milestone.html_url,html_url,str,review
573,community_activity,events,payload.issue.milestone.labels_url,labels_url,str,review


### [Section 30] Final reduced table for human selection

In [85]:
final_human_selection_df = review_df_after_strong.copy()

final_human_selection_df = final_human_selection_df.sort_values(
    ["endpoint", "relevance_score", "noise_score"],
    ascending=[True, False, True]
).reset_index(drop=True)

print("Final rows for human selection:", len(final_human_selection_df))
display(final_human_selection_df.head(100))

Final rows for human selection: 1215


,indicators,endpoint,path,key,dtype,example_value,relevance_score,measurable_score,noise_score,decision,strong_drop
0,code_quality_reliability,branches,protected,protected,bool,True,1,2,0,keep_candidate,False
1,code_quality_reliability,branches,commit,commit,dict,{'sha': '12b3264d17780b6b38f747353bc80cfd910e6...,0,0,0,review,False
2,code_quality_reliability,branches,commit.sha,sha,str,12b3264d17780b6b38f747353bc80cfd910e6dc8,0,1,0,review,False
3,code_quality_reliability,branches,name,name,str,0.19.x,0,1,0,review,False
4,community_activity,comments,created_at,created_at,str,2011-10-09T20:23:35Z,1,1,0,keep_candidate,False
...,...,...,...,...,...,...,...,...,...,...,...
95,code_quality_reliability,deployments,performed_via_github_app.events[3],[3],str,create,0,1,0,review,False
96,code_quality_reliability,deployments,performed_via_github_app.events[4],[4],str,delete,0,1,0,review,False
97,code_quality_reliability,deployments,performed_via_github_app.events[5],[5],str,deployment,0,1,0,review,False
98,code_quality_reliability,deployments,performed_via_github_app.events[6],[6],str,deployment_status,0,1,0,review,False


### [Section 31] Save results

In [ ]:
dropped_by_strong_rule_df.to_csv("output/strong_dropped_fields.csv", index=False)
final_human_selection_df.to_csv("output/final_human_selection_candidates.csv", index=False)

print("Saved:")
print("- strong_dropped_fields.csv")
print("- final_human_selection_candidates.csv")

Saved:
- strong_dropped_fields.csv
- final_human_selection_candidates.csv


### [Section 32] Text-like field pruning rules
- remove free-form text fields that are hard to quantify

In [87]:
TEXT_DROP_PATTERNS = [
    "body",
    "message",
    "description",
    "title",
    "summary",
    "note",
    "details",
    "text",
    "payload",
    "signature",
    "diff_hunk",
]

TEXT_KEEP_EXCEPTIONS = [
    "state",
    "state_reason",
    "environment",
    "license",
    "tag",
    "default_branch",
    "visibility",
    "ref",
    "ref_type",
    "task",
    "author_association",
    "type",
    "slug",
]

### [Section 33] Text-like noise classifier

In [88]:
def is_text_noise(path):
    p = str(path).lower()

    # keep short categorical/string fields
    for exc in TEXT_KEEP_EXCEPTIONS:
        if exc in p:
            return False

    # drop free-form text-like fields
    for pattern in TEXT_DROP_PATTERNS:
        if pattern in p:
            return True

    return False

### [Section 34] Apply text pruning

In [90]:
final_human_selection_df["text_drop"] = final_human_selection_df["path"].apply(
    is_text_noise
)

### [Section 35] Split after text pruning

In [91]:
after_text_pruning_df = final_human_selection_df[
    final_human_selection_df["text_drop"] == False
].copy()

text_dropped_df = final_human_selection_df[
    final_human_selection_df["text_drop"] == True
].copy()

print("Before text pruning:", len(final_human_selection_df))
print("After text pruning:", len(after_text_pruning_df))
print("Dropped by text rule:", len(text_dropped_df))

Before text pruning: 1215
After text pruning: 933
Dropped by text rule: 282


### [Section 36] Preview dropped text-like fields

In [92]:
display(
    text_dropped_df[
        ["indicators", "endpoint", "path", "key", "dtype", "decision"]
    ].head(100)
)

,indicators,endpoint,path,key,dtype,decision
8,community_activity,comments,body,body,str,review
48,"community_activity, sustainability",commits,commit.message,message,str,review
52,"community_activity, sustainability",commits,commit.verification.payload,payload,str,review
54,"community_activity, sustainability",commits,commit.verification.signature,signature,str,review
85,code_quality_reliability,deployments,description,description,NoneType,review
...,...,...,...,...,...,...
246,community_activity,events,payload.comment.position,position,int,review
247,community_activity,events,payload.comment.pull_request_review_id,pull_request_review_id,int,review
248,community_activity,events,payload.comment.reactions,reactions,dict,review
249,community_activity,events,payload.comment.reactions.+1,+1,int,review


### [Section 37] Preview reduced candidate set

In [94]:
display(after_text_pruning_df.head(100))

,indicators,endpoint,path,key,dtype,example_value,relevance_score,measurable_score,noise_score,decision,strong_drop,text_drop
0,code_quality_reliability,branches,protected,protected,bool,True,1,2,0,keep_candidate,False,False
1,code_quality_reliability,branches,commit,commit,dict,{'sha': '12b3264d17780b6b38f747353bc80cfd910e6...,0,0,0,review,False,False
2,code_quality_reliability,branches,commit.sha,sha,str,12b3264d17780b6b38f747353bc80cfd910e6dc8,0,1,0,review,False,False
3,code_quality_reliability,branches,name,name,str,0.19.x,0,1,0,review,False,False
4,community_activity,comments,created_at,created_at,str,2011-10-09T20:23:35Z,1,1,0,keep_candidate,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
102,code_quality_reliability,deployments,performed_via_github_app.id,id,int,15368,0,2,0,review,False,False
103,code_quality_reliability,deployments,performed_via_github_app.name,name,str,GitHub Actions,0,1,0,review,False,False
104,code_quality_reliability,deployments,performed_via_github_app.owner,owner,dict,"{'login': 'github', 'id': 9919, 'node_id': 'MD...",0,0,0,review,False,False
105,code_quality_reliability,deployments,performed_via_github_app.owner.id,id,int,9919,0,2,0,review,False,False


### [Section 38] Save text-pruned results

In [ ]:
text_dropped_df.to_csv("output/text_dropped_fields.csv", index=False)
after_text_pruning_df.to_csv("output/after_text_pruning_candidates.csv", index=False)

print("Saved:")
print("- text_dropped_fields.csv")
print("- after_text_pruning_candidates.csv")

Saved:
- text_dropped_fields.csv
- after_text_pruning_candidates.csv


### [Section 39] String usability pruning rules
 - remove high-cardinality / repo-specific string fields


In [98]:
STRING_KEEP_PATTERNS = [
    "state",
    "visibility",
    "environment",
    "license",
    "default_branch",
    "author_association",
    "merged",
    "protected",
    "archived",
    "locked",
    "verified",
    "login",          # aggregation seed
]

STRING_DROP_PATTERNS = [
    "name",           # branch.name / label.name / milestone.title
    "title",
    "sha",
    "ref",
    "node",
    "slug",
    "context",
    "target",
    "message",
    "tag",            # version modeling 
]

### [Section 40] String usability classifier

In [99]:
def is_unusable_string(row):

    if row["dtype"] != "str":
        return False

    path = str(row["path"]).lower()

    # keep controlled categorical / identity
    for keep_kw in STRING_KEEP_PATTERNS:
        if keep_kw in path:
            return False

    # drop high-cardinality / repo-specific tokens
    for drop_kw in STRING_DROP_PATTERNS:
        if drop_kw in path:
            return True

    return False

### [Section 41] Apply string usability pruning

In [100]:
after_text_pruning_df["string_drop"] = after_text_pruning_df.apply(
    is_unusable_string,
    axis=1
)

### [Section 42] Split after string pruning

In [101]:
after_string_pruning_df = after_text_pruning_df[
    after_text_pruning_df["string_drop"] == False
].copy()

string_dropped_df = after_text_pruning_df[
    after_text_pruning_df["string_drop"] == True
].copy()

print("Before string pruning:", len(after_text_pruning_df))
print("After string pruning:", len(after_string_pruning_df))
print("Dropped by string rule:", len(string_dropped_df))

Before string pruning: 933
After string pruning: 875
Dropped by string rule: 58


### [Section 43] Preview dropped string fields

In [102]:
display(
    string_dropped_df[
        ["indicators", "endpoint", "path", "key", "dtype"]
    ].head(100)
)

,indicators,endpoint,path,key,dtype
2,code_quality_reliability,branches,commit.sha,sha,str
3,code_quality_reliability,branches,name,name,str
43,"community_activity, sustainability",commits,commit.author.name,name,str
47,"community_activity, sustainability",commits,commit.committer.name,name,str
50,"community_activity, sustainability",commits,commit.tree.sha,sha,str
62,"community_activity, sustainability",commits,parents[0].sha,sha,str
63,"community_activity, sustainability",commits,sha,sha,str
103,code_quality_reliability,deployments,performed_via_github_app.name,name,str
131,code_quality_reliability,deployments,performed_via_github_app.slug,slug,str
132,code_quality_reliability,deployments,ref,ref,str


### [Section 44] Save results

In [104]:
after_string_pruning_df.to_csv(
    "output/after_string_pruning_candidates.csv",
    index=False
)

string_dropped_df.to_csv(
    "output/string_dropped_fields.csv",
    index=False
)

print("Saved string usability pruning results.")

Saved string usability pruning results.


### [Section 45] Measurement-axis KEEP rule

In [105]:
MEASUREMENT_KEEP_PATTERNS = [
    # time axis
    "created_at",
    "updated_at",
    "closed_at",
    "merged_at",
    "published_at",
    "verified_at",
    "pushed_at",

    # state / lifecycle
    "state",
    "draft",
    "locked",
    "protected",
    "archived",
    "disabled",
    "merged",

    # count / volume
    "comments",
    "comment_count",
    "contributions",
    "forks_count",
    "stargazers_count",
    "watchers_count",
    "open_issues_count",

    # identity aggregation
    "login",
    "author",
    "assignee",
    "reviewer",
    "committer",
    "contributor",
    "id",

    # controlled categorical
    "license",
    "environment",
    "visibility",
    "default_branch",
]

### [Section 46] Structural / Low-value DROP rule

In [106]:
STRUCTURAL_DROP_PATTERNS = [
    # dict / list containers
    "user.",
    "creator.",
    "commit.",
    "payload.",
    "reactions.",
    "labels.",
    "milestone.",
    "permissions.",
    "performed_via_github_app.",

    # profile decoration
    "site_admin",
    "user_view_type",
    "type",

    # fine-grained context
    "line",
    "position",
    "context",
    "subject_type",
]

### [Section 47] Measurement usability classifier

In [107]:
def is_measurement_drop(path):
    p = str(path).lower()

    # keep measurement axis signals
    for keep_kw in MEASUREMENT_KEEP_PATTERNS:
        if keep_kw in p:
            return False

    # drop structural / low-value signals
    for drop_kw in STRUCTURAL_DROP_PATTERNS:
        if drop_kw in p:
            return True

    return False

### [Section 48] Apply measurement pruning

In [108]:
after_string_pruning_df["measurement_drop"] = (
    after_string_pruning_df["path"].apply(is_measurement_drop)
)

### [Section 49] Split after measurement pruning

In [109]:
after_measurement_pruning_df = after_string_pruning_df[
    after_string_pruning_df["measurement_drop"] == False
].copy()

measurement_dropped_df = after_string_pruning_df[
    after_string_pruning_df["measurement_drop"] == True
].copy()

print("Before measurement pruning:", len(after_string_pruning_df))
print("After measurement pruning:", len(after_measurement_pruning_df))
print("Dropped by measurement rule:", len(measurement_dropped_df))

Before measurement pruning: 875
After measurement pruning: 644
Dropped by measurement rule: 231


### [Section 50] Source deduplication (primary endpoint rule)

In [110]:
PRIMARY_ENDPOINT_MAP = {
    "issue": "issues",
    "forkee": "forks",
    "release": "releases",
    "pull_request": "pulls",
}

def is_duplicate_source(row):
    path = str(row["path"]).lower()
    endpoint = str(row["endpoint"]).lower()

    for k, v in PRIMARY_ENDPOINT_MAP.items():
        if k in path and endpoint != v:
            return True

    return False

after_measurement_pruning_df["duplicate_source"] = (
    after_measurement_pruning_df.apply(is_duplicate_source, axis=1)
)

after_source_dedup_df = after_measurement_pruning_df[
    after_measurement_pruning_df["duplicate_source"] == False
].copy()

### [Section 51] Final preview

In [111]:
print("Final candidate raw variables:", len(after_source_dedup_df))
display(after_source_dedup_df.head(100))

Final candidate raw variables: 539


,indicators,endpoint,path,key,dtype,example_value,relevance_score,measurable_score,noise_score,decision,strong_drop,text_drop,string_drop,measurement_drop,duplicate_source
0,code_quality_reliability,branches,protected,protected,bool,True,1,2,0,keep_candidate,False,False,False,False,False
1,code_quality_reliability,branches,commit,commit,dict,{'sha': '12b3264d17780b6b38f747353bc80cfd910e6...,0,0,0,review,False,False,False,False,False
4,community_activity,comments,created_at,created_at,str,2011-10-09T20:23:35Z,1,1,0,keep_candidate,False,False,False,False,False
5,community_activity,comments,updated_at,updated_at,str,2011-10-09T20:23:35Z,1,1,0,keep_candidate,False,False,False,False,False
6,community_activity,comments,user.login,login,str,takluyver,1,1,0,keep_candidate,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
440,project_maturity,forks,private,private,bool,False,0,2,0,review,False,False,False,False,False
442,project_maturity,forks,pushed_at,pushed_at,str,2026-03-23T10:01:04Z,0,1,0,review,False,False,False,False,False
443,project_maturity,forks,size,size,int,362626,0,2,0,review,False,False,False,False,False
444,project_maturity,forks,topics,topics,list,[],0,0,0,review,False,False,False,False,False


### [Section 52] Repo-level time-series feature family rules

In [112]:

FEATURE_FAMILY_RULES = {
    "activity_volume": [
        "created_at",
        "comment_count",
        "comments",
        "contributions",
    ],
    "responsiveness": [
        "closed_at",
        "merged_at",
        "updated_at",
        "state",
    ],
    "sustainability": [
        "login",
        "contributions",
        "author",
        "committer",
        "assignee",
        "reviewer",
    ],
    "governance_workflow": [
        "protected",
        "locked",
        "draft",
        "archived",
        "disabled",
        "default_branch",
        "visibility",
        "license",
    ],
    "release_maturity": [
        "published_at",
        "tag",
        "release",
        "pushed_at",
    ],
    "popularity_ecosystem": [
        "forks_count",
        "stargazers_count",
        "watchers_count",
        "subscribers_count",
    ],
}

### [Section 53] Feature family tagger

In [113]:
def assign_feature_families(path):
    p = str(path).lower()
    matched = []

    for family, patterns in FEATURE_FAMILY_RULES.items():
        for pattern in patterns:
            if pattern in p:
                matched.append(family)
                break

    return matched

### [Section 54] Apply feature family tagging

In [114]:
after_source_dedup_df["feature_families"] = after_source_dedup_df["path"].apply(
    assign_feature_families
)

after_source_dedup_df["feature_family_str"] = after_source_dedup_df["feature_families"].apply(
    lambda x: ", ".join(x) if x else ""
)

### [Section 55] Keep only rows linked to at least one feature family

In [115]:
feature_seed_df = after_source_dedup_df[
    after_source_dedup_df["feature_families"].map(len) > 0
].copy()

feature_seed_df = feature_seed_df.sort_values(
    ["endpoint", "feature_family_str", "path"]
).reset_index(drop=True)

print("Feature seed rows:", len(feature_seed_df))
display(feature_seed_df.head(100))

Feature seed rows: 220


,indicators,endpoint,path,key,dtype,example_value,relevance_score,measurable_score,noise_score,decision,strong_drop,text_drop,string_drop,measurement_drop,duplicate_source,feature_families,feature_family_str
0,code_quality_reliability,branches,protected,protected,bool,True,1,2,0,keep_candidate,False,False,False,False,False,[governance_workflow],governance_workflow
1,community_activity,comments,created_at,created_at,str,2011-10-09T20:23:35Z,1,1,0,keep_candidate,False,False,False,False,False,[activity_volume],activity_volume
2,community_activity,comments,updated_at,updated_at,str,2011-10-09T20:23:35Z,1,1,0,keep_candidate,False,False,False,False,False,[responsiveness],responsiveness
3,community_activity,comments,author_association,author_association,str,CONTRIBUTOR,0,1,0,review,False,False,False,False,False,[sustainability],sustainability
4,community_activity,comments,user.login,login,str,takluyver,1,1,0,keep_candidate,False,False,False,False,False,[sustainability],sustainability
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,"community_activity, code_quality_reliability",pulls,base.repo.archived,archived,bool,False,0,2,0,review,False,False,False,False,False,[governance_workflow],governance_workflow
96,"community_activity, code_quality_reliability",pulls,base.repo.default_branch,default_branch,str,main,0,1,0,review,False,False,False,False,False,[governance_workflow],governance_workflow
97,"community_activity, code_quality_reliability",pulls,base.repo.disabled,disabled,bool,False,0,2,0,review,False,False,False,False,False,[governance_workflow],governance_workflow
98,"community_activity, code_quality_reliability",pulls,base.repo.license,license,dict,"{'key': 'bsd-3-clause', 'name': 'BSD 3-Clause ...",1,0,0,review,False,False,False,False,False,[governance_workflow],governance_workflow


### [Section 56] Reorder columns for easier review

In [116]:
feature_seed_df = feature_seed_df[
    [
        "indicators",
        "feature_family_str",
        "endpoint",
        "path",
        "key",
        "dtype",
        "example_value",
        "relevance_score",
        "measurable_score",
        "noise_score",
        "decision",
    ]
]

display(feature_seed_df.head(100))

,indicators,feature_family_str,endpoint,path,key,dtype,example_value,relevance_score,measurable_score,noise_score,decision
0,code_quality_reliability,governance_workflow,branches,protected,protected,bool,True,1,2,0,keep_candidate
1,community_activity,activity_volume,comments,created_at,created_at,str,2011-10-09T20:23:35Z,1,1,0,keep_candidate
2,community_activity,responsiveness,comments,updated_at,updated_at,str,2011-10-09T20:23:35Z,1,1,0,keep_candidate
3,community_activity,sustainability,comments,author_association,author_association,str,CONTRIBUTOR,0,1,0,review
4,community_activity,sustainability,comments,user.login,login,str,takluyver,1,1,0,keep_candidate
...,...,...,...,...,...,...,...,...,...,...,...
95,"community_activity, code_quality_reliability",governance_workflow,pulls,base.repo.archived,archived,bool,False,0,2,0,review
96,"community_activity, code_quality_reliability",governance_workflow,pulls,base.repo.default_branch,default_branch,str,main,0,1,0,review
97,"community_activity, code_quality_reliability",governance_workflow,pulls,base.repo.disabled,disabled,bool,False,0,2,0,review
98,"community_activity, code_quality_reliability",governance_workflow,pulls,base.repo.license,license,dict,"{'key': 'bsd-3-clause', 'name': 'BSD 3-Clause ...",1,0,0,review


### [Section 57] Feature family summary

In [117]:
family_summary_df = (
    feature_seed_df["feature_family_str"]
    .value_counts()
    .reset_index()
)

family_summary_df.columns = ["feature_family_str", "count"]

display(family_summary_df)

,feature_family_str,count
0,sustainability,81
1,responsiveness,50
2,governance_workflow,38
3,activity_volume,31
4,popularity_ecosystem,13
5,release_maturity,6
6,"activity_volume, sustainability",1


### [Section 58] Endpoint × feature family summary

In [118]:
endpoint_family_summary_df = (
    feature_seed_df.groupby(["endpoint", "feature_family_str"])
    .size()
    .reset_index(name="count")
    .sort_values(["endpoint", "count"], ascending=[True, False])
    .reset_index(drop=True)
)

display(endpoint_family_summary_df)

,endpoint,feature_family_str,count
0,branches,governance_workflow,1
1,comments,sustainability,2
2,comments,activity_volume,1
3,comments,responsiveness,1
4,commits,sustainability,18
5,commits,activity_volume,1
6,contributors,"activity_volume, sustainability",1
7,contributors,sustainability,1
8,deployments,activity_volume,2
9,deployments,responsiveness,2


### [Section 59] Save repo-level feature seed table

In [119]:
feature_seed_df.to_csv("output/repo_level_feature_seeds.csv", index=False)
family_summary_df.to_csv("output/feature_family_summary.csv", index=False)
endpoint_family_summary_df.to_csv("output/endpoint_feature_family_summary.csv", index=False)

print("Saved:")
print("- repo_level_feature_seeds.csv")
print("- feature_family_summary.csv")
print("- endpoint_feature_family_summary.csv")

Saved:
- repo_level_feature_seeds.csv
- feature_family_summary.csv
- endpoint_feature_family_summary.csv


### [Section 60] Time-series feature generation score rule

In [120]:
def feature_generation_score(path, dtype):
    p = str(path).lower()

    # Temporal anchor → strongest signal
    if any(t in p for t in [
        "created_at",
        "updated_at",
        "closed_at",
        "merged_at",
        "published_at",
        "pushed_at",
        "date"
    ]):
        return 3

    # Contributor aggregation seed
    if any(t in p for t in [
        "login",
        "contributor",
        "committer",
        "author"
    ]):
        return 2

    # Volume / intensity numeric signal
    if any(t in p for t in [
        "count",
        "comments",
        "contributions",
        "forks",
        "stargazers",
        "subscribers",
        "watchers",
        "open_issues"
    ]):
        return 2

    # Lifecycle categorical
    if any(t in p for t in [
        "state",
        "merged",
        "draft",
        "locked",
        "protected",
        "archived",
        "disabled"
    ]):
        return 1

    # Governance static categorical
    if any(t in p for t in [
        "license",
        "visibility",
        "default_branch",
        "environment"
    ]):
        return 1

    return 0

### [Section 61] Apply feature generation scoring

In [121]:
feature_seed_df["feature_gen_score"] = feature_seed_df.apply(
    lambda r: feature_generation_score(r["path"], r["dtype"]),
    axis=1
)

### [Section 62] Keep only strong time-series signals

In [122]:
ts_raw_signal_df = feature_seed_df[
    feature_seed_df["feature_gen_score"] >= 2
].copy()

print("Remaining raw signals for TS modeling:", len(ts_raw_signal_df))
display(ts_raw_signal_df.head(100))

Remaining raw signals for TS modeling: 147


,indicators,feature_family_str,endpoint,path,key,dtype,example_value,relevance_score,measurable_score,noise_score,decision,feature_gen_score
1,community_activity,activity_volume,comments,created_at,created_at,str,2011-10-09T20:23:35Z,1,1,0,keep_candidate,3
2,community_activity,responsiveness,comments,updated_at,updated_at,str,2011-10-09T20:23:35Z,1,1,0,keep_candidate,3
3,community_activity,sustainability,comments,author_association,author_association,str,CONTRIBUTOR,0,1,0,review,2
4,community_activity,sustainability,comments,user.login,login,str,takluyver,1,1,0,keep_candidate,2
5,"community_activity, sustainability",activity_volume,commits,commit.comment_count,comment_count,int,0,1,2,0,keep_candidate,2
...,...,...,...,...,...,...,...,...,...,...,...,...
147,"sustainability, project_maturity",activity_volume,releases,assets[1].created_at,created_at,str,2016-05-03T21:10:33Z,1,1,0,keep_candidate,3
148,"sustainability, project_maturity",activity_volume,releases,assets[2].created_at,created_at,str,2016-05-03T21:10:33Z,1,1,0,keep_candidate,3
149,"sustainability, project_maturity",activity_volume,releases,assets[3].created_at,created_at,str,2016-05-03T21:10:33Z,1,1,0,keep_candidate,3
150,"sustainability, project_maturity",activity_volume,releases,assets[4].created_at,created_at,str,2016-05-03T21:10:33Z,1,1,0,keep_candidate,3


### [Section 63] Drop redundant identity attributes

In [123]:
ts_raw_signal_df = ts_raw_signal_df[
    ~ts_raw_signal_df["path"].str.contains(
        "site_admin|type|user_view_type",
        case=False,
        na=False
    )
]

### [Section 64] Endpoint-level deduplication 

In [124]:
PRIMARY_SIGNAL_ENDPOINT = {
    "created_at": ["commits", "issues", "pulls", "comments"],
    "login": ["contributors"],
    "contributions": ["contributors"],
    "forks_count": ["repo"],
    "stargazers_count": ["repo"],
}

def strong_dedup(row):
    p = str(row["path"]).lower()
    ep = str(row["endpoint"]).lower()

    for signal, preferred_eps in PRIMARY_SIGNAL_ENDPOINT.items():
        if signal in p and ep not in preferred_eps:
            return True
    return False

ts_raw_signal_df["strong_dup"] = ts_raw_signal_df.apply(
    strong_dedup,
    axis=1
)

ts_raw_signal_df = ts_raw_signal_df[
    ts_raw_signal_df["strong_dup"] == False
].copy()

### [Section 65] Final repo-level TS raw signal set

In [125]:
print("Final TS raw signal count:", len(ts_raw_signal_df))

ts_raw_signal_df = ts_raw_signal_df.sort_values(
    ["endpoint", "path"]
).reset_index(drop=True)

display(ts_raw_signal_df.head(200))

Final TS raw signal count: 72


,indicators,feature_family_str,endpoint,path,key,dtype,example_value,relevance_score,measurable_score,noise_score,decision,feature_gen_score,strong_dup
0,community_activity,sustainability,comments,author_association,author_association,str,CONTRIBUTOR,0,1,0,review,2,False
1,community_activity,activity_volume,comments,created_at,created_at,str,2011-10-09T20:23:35Z,1,1,0,keep_candidate,3,False
2,community_activity,responsiveness,comments,updated_at,updated_at,str,2011-10-09T20:23:35Z,1,1,0,keep_candidate,3,False
3,"community_activity, sustainability",sustainability,commits,author,author,dict,"{'login': 'kimimgo', 'id': 21175731, 'node_id'...",0,0,0,review,2,False
4,"community_activity, sustainability",sustainability,commits,author.id,id,int,21175731,0,2,0,review,2,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,"legal_governance, project_maturity",popularity_ecosystem,repo,stargazers_count,stargazers_count,int,48221,1,2,0,keep_candidate,2,False
68,"legal_governance, project_maturity",popularity_ecosystem,repo,subscribers_count,subscribers_count,int,1105,1,2,0,keep_candidate,2,False
69,"legal_governance, project_maturity",responsiveness,repo,updated_at,updated_at,str,2026-03-23T07:44:47Z,1,1,0,keep_candidate,3,False
70,"legal_governance, project_maturity",popularity_ecosystem,repo,watchers_count,watchers_count,int,48221,0,2,0,review,2,False


### [Section 66] Save final TS raw signal pool

In [ ]:
ts_raw_signal_df.to_csv(
    "output/final_repo_ts_raw_signals.csv",
    index=False
)

print("Saved final_repo_ts_raw_signals.csv")

Saved final_repo_ts_raw_signals.csv
